# Phase 2: Trajectory Prediction with Auxiliary Depth Estimation

# 🧭 Introduction

"""
Welcome to **Phase 2** of the DLAV Projec! 🚗💨

In this phase, you'll work with a more challenging dataset that includes:
- RGB **camera images**
- Ground-truth **depth maps**
- Ground-truth **semantic segmentation** labels

Your goal is still to predict the **future trajectory** of the self-driving car (SDC), but you now have more tools at your disposal! 🎯

Here, we provide an example where **depth estimation** is used as an auxiliary task to improve trajectory prediction.

However, you're **free to explore** other auxiliary tasks (e.g., using semantic labels), different loss functions, data augmentations, or better architectures! 💡

This notebook will walk you through loading the dataset, building a model, training with and without the auxiliary task, and visualizing results.
"""

In [ ]:
# Data already downloaded under ../data/ — uncomment to re-download.
# !pip install -q gdown
# import gdown, zipfile, os
# os.makedirs("../data", exist_ok=True)
# for fid, name in [
#     ("1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr", "dlav_train.zip"),
#     ("1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu", "dlav_val.zip"),
#     ("1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV", "dlav_test_public.zip"),
# ]:
#     gdown.download(f"https://drive.google.com/uc?id={fid}", f"../data/{name}", quiet=False)
#     with zipfile.ZipFile(f"../data/{name}") as z:
#         z.extractall("../data/")

## 📂 The Dataset

We are now working with a richer dataset that includes not just images and trajectories,
but also **depth maps** (and semantic segmentation labels, though unused in this example).

The data is stored in `.pkl` files and each file contains:
- `camera`: RGB image (shape: H x W x 3)
- `sdc_history_feature`: the past trajectory of the car
- `sdc_future_feature`: the future trajectory to predict
- `depth`: ground truth depth map (shape: H x W x 1)

We'll define a `DrivingDataset` class to load and return these tensors in a format our model can work with.

In [1]:
import os
import torch
import pickle
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import random

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

COMMAND_MAP = {'forward': 0, 'left': 1, 'right': 2}
COMMAND_FLIP = {0: 0, 1: 2, 2: 1}  # forward→forward, left↔right

class DrivingDataset(Dataset):
    def __init__(self, file_list, test=False, augment=False):
        self.samples = file_list
        self.test = test
        self.augment = augment
        self.img_norm = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.color_jitter = T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        with open(self.samples[idx], 'rb') as f:
            data = pickle.load(f)

        camera = torch.FloatTensor(data['camera']).permute(2, 0, 1) / 255.0   # [3, 200, 300]
        depth  = torch.FloatTensor(data['depth']).permute(2, 0, 1)             # [1, 200, 300]
        history = torch.FloatTensor(data['sdc_history_feature'])
        command = COMMAND_MAP[data['driving_command']]
        future = None if self.test else torch.FloatTensor(data['sdc_future_feature'])

        if self.augment:
            # Color jitter — camera only
            if random.random() < 0.5:
                camera = self.color_jitter(camera)

            # Horizontal flip — coherent across camera, depth, trajectories, command
            if random.random() < 0.5:
                camera = TF.hflip(camera)
                depth  = TF.hflip(depth)
                history = history.clone()
                history[:, 0] *= -1   # negate x
                history[:, 2] *= -1   # negate heading
                if future is not None:
                    future = future.clone()
                    future[:, 0] *= -1
                    future[:, 2] *= -1
                command = COMMAND_FLIP[command]

            # Speed scaling — trajectory only
            if random.random() < 0.5:
                scale = random.uniform(0.8, 1.2)
                history = history.clone()
                history[:, :2] *= scale
                if future is not None:
                    future = future.clone()
                    future[:, :2] *= scale

        # Resize camera to 224x224 for DINOv3 (depth stays 200x300 as supervision target)
        camera = TF.resize(camera, [224, 224], antialias=True)
        camera = self.img_norm(camera)
        command = torch.tensor(command, dtype=torch.long)

        out = {'camera': camera, 'history': history, 'command': command, 'depth': depth}
        if future is not None:
            out['future'] = future
        return out

## 🧠 The Model: Trajectory + Depth Prediction

We've extended our trajectory prediction model to optionally include a **depth estimation decoder**.

Why?
- Predicting depth helps the model **learn richer visual features** from the camera input.
- This acts as a form of **multi-task learning**, where learning to estimate depth reinforces scene understanding, ultimately leading to better trajectory predictions.
- This can be especially useful in complex environments with occlusions or sharp turns.

The model has:
- A CNN backbone to extract features from the image
- An MLP to process historical trajectory features
- A trajectory decoder to predict future coordinates
- (Optionally) A depth decoder to predict dense depth maps

This auxiliary task is enabled by setting `use_depth_aux=True`.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DINO_REPO_DIR = "../external/dinov3"
DINO_WEIGHTS  = "../external/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"

class DrivingPlanner(nn.Module):
    """
    Phase 2 architecture:
      - DINOv3 ViT-S/16 backbone (frozen) → 196 patch tokens (14×14) + CLS
      - GRU encodes 21-step history → 1 token
      - Embedding for driving command → 1 token
      - Projected memory tokens [patches + hist + cmd] feed a Transformer decoder
      - 60 learnable waypoint queries → (x, y) per future step
      - Optional depth head: projected patch tokens → conv upsampler → (200, 300, 1)
        Depth loss back-propagates into the patch projection (shared with trajectory),
        forcing it to keep spatial information useful for navigation.
    """
    def __init__(
        self,
        use_depth_aux=False,
        cmd_embed_dim=64,
        gru_hidden=256,
        fusion_dim=256,
        nhead=4,
        num_decoder_layers=3,
        dropout=0.1,
    ):
        super().__init__()
        self.use_depth_aux = use_depth_aux

        # DINOv3 backbone (frozen)
        self.dino = torch.hub.load(DINO_REPO_DIR, "dinov3_vits16", source="local", weights=DINO_WEIGHTS)
        for p in self.dino.parameters():
            p.requires_grad = False
        dino_dim = 384
        self.grid = 14  # 224 / 16
        self.num_patches = self.grid * self.grid  # 196

        # History encoder
        self.gru = nn.GRU(input_size=3, hidden_size=gru_hidden, num_layers=2,
                          batch_first=True, dropout=dropout)

        # Command embedding
        self.cmd_embed = nn.Embedding(3, cmd_embed_dim)

        # Modality projections → fusion_dim
        self.patch_proj = nn.Sequential(nn.Linear(dino_dim,     fusion_dim), nn.LayerNorm(fusion_dim))
        self.hist_proj  = nn.Sequential(nn.Linear(gru_hidden,   fusion_dim), nn.LayerNorm(fusion_dim))
        self.cmd_proj   = nn.Sequential(nn.Linear(cmd_embed_dim, fusion_dim), nn.LayerNorm(fusion_dim))

        # Token type embedding (so decoder can tell patches / hist / cmd apart)
        self.token_type_embed = nn.Embedding(3, fusion_dim)  # 0=patch, 1=hist, 2=cmd

        # Learnable waypoint queries (one per future timestep)
        self.waypoint_queries = nn.Parameter(torch.randn(60, fusion_dim) * 0.02)

        # Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=fusion_dim, nhead=nhead,
            dim_feedforward=fusion_dim * 4,
            dropout=dropout, batch_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)

        # Trajectory head — each decoded query → (x, y)
        self.traj_head = nn.Linear(fusion_dim, 2)

        # Depth head (optional). Takes projected patches → 4× upsample to 224×224 → bilinear to 200×300
        if use_depth_aux:
            self.depth_head = nn.Sequential(
                nn.ConvTranspose2d(fusion_dim, 128, kernel_size=4, stride=2, padding=1),  # 14→28
                nn.GELU(),
                nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),           # 28→56
                nn.GELU(),
                nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),            # 56→112
                nn.GELU(),
                nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),            # 112→224
                nn.GELU(),
                nn.Conv2d(16, 1, kernel_size=3, padding=1),
                nn.Upsample(size=(200, 300), mode='bilinear', align_corners=False),
            )

    def _patch_tokens(self, camera):
        """Returns DINOv3 patch tokens [B, num_patches, dino_dim] (no grad through backbone)."""
        with torch.no_grad():
            feats = self.dino.forward_features(camera)
        return feats['x_norm_patchtokens']

    def forward(self, camera, history, command):
        B = camera.size(0)

        patch_tokens = self._patch_tokens(camera)         # [B, 196, 384]

        _, h_n = self.gru(history)
        hist_feat = h_n[-1]                                # [B, gru_hidden]
        cmd_feat  = self.cmd_embed(command)                # [B, cmd_embed_dim]

        # Project all modalities to common dim
        patch_tok = self.patch_proj(patch_tokens)          # [B, 196, fd]
        hist_tok  = self.hist_proj(hist_feat).unsqueeze(1) # [B, 1, fd]
        cmd_tok   = self.cmd_proj(cmd_feat).unsqueeze(1)   # [B, 1, fd]

        # Token type embeddings
        type_ids = torch.cat([
            torch.zeros(self.num_patches, dtype=torch.long, device=camera.device),
            torch.ones(1, dtype=torch.long, device=camera.device),
            torch.full((1,), 2, dtype=torch.long, device=camera.device),
        ])
        type_emb = self.token_type_embed(type_ids).unsqueeze(0)   # [1, 198, fd]

        memory = torch.cat([patch_tok, hist_tok, cmd_tok], dim=1) + type_emb  # [B, 198, fd]

        # Decode waypoints via cross-attention
        queries = self.waypoint_queries.unsqueeze(0).expand(B, -1, -1)        # [B, 60, fd]
        decoded = self.transformer_decoder(queries, memory)                    # [B, 60, fd]
        traj = self.traj_head(decoded)                                         # [B, 60, 2]

        # Optional depth prediction from projected patches
        depth_out = None
        if self.use_depth_aux:
            patch_2d = patch_tok.transpose(1, 2).reshape(B, -1, self.grid, self.grid)  # [B, fd, 14, 14]
            depth_out = self.depth_head(patch_2d).permute(0, 2, 3, 1)                  # [B, 200, 300, 1]

        return traj, depth_out

    def trainable_parameters(self):
        return (p for p in self.parameters() if p.requires_grad)

## 🏋️ Training with Auxiliary Loss

The training loop is similar to Phase 1 — except now, if enabled, we also compute a loss on the predicted **depth map**.

We define:
- `trajectory_loss` as standard MSE between predicted and true future trajectory
- `depth_loss` as L1 loss between predicted and ground truth depth

Total loss = `trajectory_loss + lambda * depth_loss`

This helps guide the model to learn better representations from visual input. The weight `lambda` is a hyperparameter you can tune!

In [3]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt


def smoothness_loss(pred):
    """pred: [B, 60, 2] → (mean velocity norm, mean acceleration norm)."""
    delta = pred[:, 1:] - pred[:, :-1]
    accel = delta[:, 1:] - delta[:, :-1]
    return delta.norm(dim=-1).mean(), accel.norm(dim=-1).mean()


class Logger:
    def __init__(self):
        self.train_losses, self.val_ades, self.val_fdes = [], [], []
        self.depth_losses, self.lrs = [], []

    def log(self, train_loss, ade, fde, lr, depth_loss=None):
        self.train_losses.append(train_loss)
        self.val_ades.append(ade); self.val_fdes.append(fde); self.lrs.append(lr)
        if depth_loss is not None:
            self.depth_losses.append(depth_loss)

    def plot(self):
        has_dep = bool(self.depth_losses)
        n = 3 + has_dep
        fig, ax = plt.subplots(1, n, figsize=(5*n, 4))
        ax[0].plot(self.train_losses); ax[0].set_title('Train Traj Loss'); ax[0].set_xlabel('Epoch')
        ax[1].plot(self.val_ades, label='ADE'); ax[1].plot(self.val_fdes, label='FDE')
        ax[1].set_title('Validation'); ax[1].set_xlabel('Epoch'); ax[1].legend()
        ax[2].plot(self.lrs); ax[2].set_title('Learning Rate'); ax[2].set_xlabel('Epoch')
        if has_dep:
            ax[3].plot(self.depth_losses); ax[3].set_title('Train Depth Loss'); ax[3].set_xlabel('Epoch')
        plt.tight_layout(); plt.show()


def train_one_epoch(model, loader, optimizer, device, *,
                    use_depth_aux, lambda_depth, lambda_smooth, lambda_jerk, grad_clip):
    model.train()
    tot_traj, tot_dep, n = 0.0, 0.0, 0
    for batch in loader:
        cam = batch['camera'].to(device)
        hist = batch['history'].to(device)
        cmd = batch['command'].to(device)
        fut = batch['future'].to(device)
        dep = batch['depth'].to(device)

        optimizer.zero_grad()
        traj_pred, dep_pred = model(cam, hist, cmd)

        traj_loss = F.smooth_l1_loss(traj_pred, fut[..., :2])
        smooth, jerk = smoothness_loss(traj_pred)
        loss = traj_loss + lambda_smooth * smooth + lambda_jerk * jerk

        if use_depth_aux:
            # GT depth: [B, 1, 200, 300] → match prediction shape [B, 200, 300, 1]
            dep_gt = dep.permute(0, 2, 3, 1)
            dep_loss = F.l1_loss(dep_pred, dep_gt)
            loss = loss + lambda_depth * dep_loss
            tot_dep += dep_loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        tot_traj += traj_loss.item()
        n += 1

    return tot_traj / n, (tot_dep / n if use_depth_aux else None)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    ades, fdes = [], []
    for batch in loader:
        cam = batch['camera'].to(device)
        hist = batch['history'].to(device)
        cmd = batch['command'].to(device)
        fut = batch['future'].to(device)
        traj_pred, _ = model(cam, hist, cmd)
        ades.append(torch.norm(traj_pred - fut[..., :2], dim=-1).mean().item())
        fdes.append(torch.norm(traj_pred[:, -1] - fut[:, -1, :2], dim=-1).mean().item())
    return float(np.mean(ades)), float(np.mean(fdes))


def train(model, train_loader, val_loader, optimizer, scheduler=None, num_epochs=50,
          use_depth_aux=False, lambda_depth=0.1, lambda_smooth=0.1, lambda_jerk=0.05, grad_clip=1.0):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    model = model.to(device)
    logger = Logger()

    for epoch in range(num_epochs):
        train_traj, train_dep = train_one_epoch(
            model, train_loader, optimizer, device,
            use_depth_aux=use_depth_aux, lambda_depth=lambda_depth,
            lambda_smooth=lambda_smooth, lambda_jerk=lambda_jerk, grad_clip=grad_clip,
        )
        if scheduler is not None:
            scheduler.step()
        ade, fde = validate(model, val_loader, device)
        lr = optimizer.param_groups[0]['lr']
        logger.log(train_traj, ade, fde, lr, train_dep)

        msg = f"Epoch {epoch+1}/{num_epochs} | Train Traj: {train_traj:.4f} | ADE: {ade:.4f} | FDE: {fde:.4f}"
        if use_depth_aux:
            msg += f" | Depth: {train_dep:.4f}"
        msg += f" | LR: {lr:.2e}"
        print(msg)

    logger.plot()
    return logger

In [4]:
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import os

DATA_DIR = "../data"
train_data_dir = f"{DATA_DIR}/train"
val_data_dir   = f"{DATA_DIR}/val"

train_files = [os.path.join(train_data_dir, f) for f in os.listdir(train_data_dir) if f.endswith('.pkl')]
val_files   = [os.path.join(val_data_dir,   f) for f in os.listdir(val_data_dir)   if f.endswith('.pkl')]

# Weighted sampler: upweight stopped/slow samples so the model doesn't always predict motion
print("Computing sample weights...")
speeds = []
for f in train_files:
    with open(f, 'rb') as fp:
        d = pickle.load(fp)
    hist = d['sdc_history_feature']
    speed = np.linalg.norm(np.diff(hist[:, :2], axis=0), axis=1).mean()
    speeds.append(speed)
speeds = np.array(speeds)
weights = np.where(speeds < 0.5, 5.0, 1.0)
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
print(f"Stopped samples (speed < 0.5): {(speeds < 0.5).sum()} / {len(speeds)}")

train_dataset = DrivingDataset(train_files, augment=True)
val_dataset   = DrivingDataset(val_files,   augment=False)

# num_workers=0 to avoid multiprocessing issues with Jupyter on Python 3.14
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, num_workers=0)

Computing sample weights...
Stopped samples (speed < 0.5): 2878 / 5000


## 🔍 Let's Compare Two Settings

We'll now train and evaluate the model in **two modes**:

1. **Without auxiliary task** — the model only predicts the trajectory.
2. **With depth auxiliary task** — the model also predicts a depth map, which helps it learn better visual features.

By comparing the results (ADE, FDE, and Trajectory MSE), you'll see the benefit of multi-task learning in action! 🚀

In [ ]:
NUM_EPOCHS = 200
WARMUP = 5

use_depth_aux = False
model_no_aux = DrivingPlanner(use_depth_aux=use_depth_aux)

optimizer = optim.AdamW(model_no_aux.trainable_parameters(), lr=3e-4, weight_decay=1e-4)
warmup  = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP)
cosine  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP, eta_min=1e-6)
scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP])

logger_no_aux = train(
    model_no_aux, train_loader, val_loader, optimizer,
    scheduler=scheduler, num_epochs=NUM_EPOCHS, use_depth_aux=use_depth_aux,
)

Device: cuda
Epoch 1/200 | Train Traj: 4.2126 | ADE: 14.7155 | FDE: 29.5241 | LR: 6.24e-05
Epoch 2/200 | Train Traj: 3.2765 | ADE: 10.0622 | FDE: 23.9202 | LR: 1.22e-04
Epoch 3/200 | Train Traj: 2.6496 | ADE: 8.8267 | FDE: 22.3769 | LR: 1.81e-04
Epoch 4/200 | Train Traj: 2.3693 | ADE: 8.3418 | FDE: 20.8650 | LR: 2.41e-04
Epoch 5/200 | Train Traj: 2.1584 | ADE: 6.9634 | FDE: 19.0146 | LR: 3.00e-04
Epoch 6/200 | Train Traj: 1.9689 | ADE: 6.6780 | FDE: 18.4986 | LR: 3.00e-04
Epoch 7/200 | Train Traj: 1.7930 | ADE: 5.7251 | FDE: 15.4405 | LR: 3.00e-04
Epoch 8/200 | Train Traj: 1.6934 | ADE: 5.1357 | FDE: 14.4684 | LR: 3.00e-04
Epoch 9/200 | Train Traj: 1.4985 | ADE: 5.0978 | FDE: 13.9816 | LR: 3.00e-04
Epoch 10/200 | Train Traj: 1.4316 | ADE: 4.0820 | FDE: 12.1688 | LR: 3.00e-04
Epoch 11/200 | Train Traj: 1.3796 | ADE: 3.8208 | FDE: 11.9726 | LR: 2.99e-04
Epoch 12/200 | Train Traj: 1.3794 | ADE: 3.4190 | FDE: 11.0886 | LR: 2.99e-04


In [ ]:
use_depth_aux = True
model_with_aux = DrivingPlanner(use_depth_aux=use_depth_aux)

optimizer = optim.AdamW(model_with_aux.trainable_parameters(), lr=3e-4, weight_decay=1e-4)
warmup  = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP)
cosine  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP, eta_min=1e-6)
scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP])

logger_with_aux = train(
    model_with_aux, train_loader, val_loader, optimizer,
    scheduler=scheduler, num_epochs=NUM_EPOCHS, use_depth_aux=use_depth_aux,
    lambda_depth=0.1,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ade, fde = validate(model_no_aux, val_loader, device)
print(f"Without depth aux: ADE={ade:.4f}, FDE={fde:.4f}")

ade, fde = validate(model_with_aux, val_loader, device)
print(f"With depth aux:    ADE={ade:.4f}, FDE={fde:.4f}")

# Save checkpoints
os.makedirs("../models", exist_ok=True)
torch.save(model_no_aux.state_dict(),   "../models/phase2_no_aux.pth")
torch.save(model_with_aux.state_dict(), "../models/phase2_with_aux.pth")
print("Checkpoints saved to ../models/")

## 🔍 Final Visualization and Comparison

Now that we’ve trained two models — one **with** the depth auxiliary task and one **without** — let’s visualize and compare their predictions.

We’ll show:
1. The **camera image** from selected validation examples
2. The **past trajectory**, **ground-truth future**, and **predicted future** trajectory
3. The **predicted vs. ground-truth depth maps** (only for the model trained with the auxiliary task)

These visualizations help us understand:
- Does the predicted trajectory better match the future when the depth task is included?
- Is the predicted depth map reasonably accurate?

Let’s see the difference! 📈

In [ ]:
import matplotlib.pyplot as plt
import random
random.seed(40)

# ImageNet stats — used to denormalize the camera tensor for display
_MEAN = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
_STD  = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)


def visualize_comparison(val_loader, model_no_aux, model_with_aux, device):
    model_no_aux.eval(); model_with_aux.eval()
    batch = next(iter(val_loader))

    camera  = batch['camera'].to(device)
    history = batch['history'].to(device)
    command = batch['command'].to(device)
    future  = batch['future'].to(device)
    depth   = batch['depth'].to(device)

    with torch.no_grad():
        pred_no_aux,   _          = model_no_aux(camera, history, command)
        pred_with_aux, pred_depth = model_with_aux(camera, history, command)

    camera = camera.cpu().numpy(); history = history.cpu().numpy(); future = future.cpu().numpy()
    pred_no_aux   = pred_no_aux.cpu().numpy()
    pred_with_aux = pred_with_aux.cpu().numpy()
    depth         = depth.cpu().numpy()
    pred_depth    = pred_depth.cpu().numpy() if pred_depth is not None else None

    k = 4
    indices = random.choices(np.arange(len(camera)), k=k)

    # Camera inputs (denormalized)
    fig, ax = plt.subplots(1, k, figsize=(4*k, 4))
    for i, idx in enumerate(indices):
        img = (camera[idx] * _STD + _MEAN).transpose(1, 2, 0).clip(0, 1)
        ax[i].imshow(img); ax[i].set_title(f"Example {i+1}"); ax[i].axis("off")
    plt.suptitle("Camera Inputs"); plt.tight_layout(); plt.show()

    # Trajectories
    fig, ax = plt.subplots(2, k, figsize=(4*k, 8))
    for i, idx in enumerate(indices):
        for row, (pred, title, color) in enumerate([
            (pred_no_aux,   "No Depth Aux",   'red'),
            (pred_with_aux, "With Depth Aux", 'blue'),
        ]):
            ax[row, i].plot(history[idx, :, 0], history[idx, :, 1], 'o-', color='gold',  markersize=4, label='Past')
            ax[row, i].plot(future[idx, :, 0],  future[idx, :, 1],  'o-', color='green', markersize=4, label='GT')
            ax[row, i].plot(pred[idx, :, 0],    pred[idx, :, 1],    'o-', color=color,   markersize=4, label='Pred')
            ax[row, i].set_title(title); ax[row, i].axis("equal"); ax[row, i].legend(fontsize=8)
    plt.tight_layout(); plt.show()

    # Depth
    if pred_depth is not None:
        fig, ax = plt.subplots(2, k, figsize=(4*k, 6))
        for i, idx in enumerate(indices):
            ax[0, i].imshow(depth[idx, 0],          cmap='viridis'); ax[0, i].set_title("GT Depth");   ax[0, i].axis("off")
            ax[1, i].imshow(pred_depth[idx, :, :, 0], cmap='viridis'); ax[1, i].set_title("Pred Depth"); ax[1, i].axis("off")
        plt.suptitle("Depth Estimation (model with aux)", y=1.02); plt.tight_layout(); plt.show()


visualize_comparison(val_loader, model_no_aux, model_with_aux,
                     device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

Now we run our model on the test set once, to get the plan of our model and save it for submission. Notice that the ground truth plans are removed for the test set, so you can not calculate the ADE metric on the test set yourself, and need to submit it to the leader board. By running the last cell, you'll be able to see a csv file called submission_phase2.csv by clicking on the folder icon on the left. Download it and submit it to the leaderboard to get your score.

In [ ]:
with open(f"../data/test_public/0.pkl", "rb") as f:
    data = pickle.load(f)
print(data.keys())
# Note the absence of sdc_future_feature

In [ ]:
import pandas as pd

test_data_dir = "../data/test_public"
test_files = [
    os.path.join(test_data_dir, fn)
    for fn in sorted(
        [f for f in os.listdir(test_data_dir) if f.endswith(".pkl")],
        key=lambda fn: int(os.path.splitext(fn)[0]),
    )
]

test_dataset = DrivingDataset(test_files, test=True)
test_loader  = DataLoader(test_dataset, batch_size=128, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_with_aux.eval()
all_plans = []
with torch.no_grad():
    for batch in test_loader:
        cam = batch['camera'].to(device)
        hist = batch['history'].to(device)
        cmd  = batch['command'].to(device)
        pred, _ = model_with_aux(cam, hist, cmd)
        all_plans.append(pred.cpu().numpy())
all_plans = np.concatenate(all_plans, axis=0)  # [N, 60, 2]

total_samples, T, _ = all_plans.shape
pred_xy_flat = all_plans.reshape(total_samples, T * 2)

df = pd.DataFrame(pred_xy_flat)
df.insert(0, "id", np.arange(total_samples))
cols = ["id"]
for t in range(1, T + 1):
    cols += [f"x_{t}", f"y_{t}"]
df.columns = cols

os.makedirs("../submissions", exist_ok=True)
df.to_csv("../submissions/submission_phase2.csv", index=False)
print(f"Shape: {df.shape}")